In [3]:
"""
Extract rainfall, soil moisture, temperature, VPD and LST per 16-day window at the sample points.
Windows use the SAME grid as the MODIS label: window k starts on DOY 1 + 16k (the last one is clipped at 31 Dec).
All 23 windows (k = 0..22) are exported so lags and SPI accumulation can cross months
"""
import argparse
import os
import sys
import urllib.request
from datetime import date, timedelta

import ee
import pandas as pd

# ------------------------- CONFIG -------------------------
POINTS_ASSET = "projects/ld-slm/assets/crop_stress/sample_points"  
DRIVE_FOLDER = "crop_stress_climate"
DEFAULT_YEARS = list(range(2001, 2026))
ALL_K = list(range(0, 23))      
NODATA = -9999
SEED = 42
N_CHUNKS = 20                  
VARS = ["chirps", "dry", "sm1", "sm2", "t2m", "vpd", "lst"]


# ------------------------- windows -------------------------
def window_dates(year, k):
    start = date(year, 1, 1) + timedelta(days=16 * k)
    end = min(start + timedelta(days=16), date(year + 1, 1, 1))
    return start.isoformat(), end.isoformat()


def band_names(ks=ALL_K):
    return [f"{v}_k{k:02d}" for k in ks for v in VARS]


# ------------------------- collections -------------------------
def chirps_ic():
    return ee.ImageCollection("UCSB-CHG/CHIRPS/DAILY").select("precipitation")


def era5_ic():
    ic = ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")

    def prep(img):
        t = img.select("temperature_2m").subtract(273.15)
        td = img.select("dewpoint_temperature_2m").subtract(273.15)
        es = t.multiply(17.27).divide(t.add(237.3)).exp().multiply(0.6108)
        ea = td.multiply(17.27).divide(td.add(237.3)).exp().multiply(0.6108)
        vpd = es.subtract(ea).max(0).rename("vpd")
        out = (
            img.select(["volumetric_soil_water_layer_1", "volumetric_soil_water_layer_2"])
            .rename(["sm1", "sm2"])
            .addBands(t.rename("t2m"))
            .addBands(vpd)
        )
        return out.copyProperties(img, ["system:time_start"])

    return ic.map(prep)


def lst_ic():
    ic = ee.ImageCollection("MODIS/061/MOD11A2")

    def prep(img):
        raw = img.select("LST_Day_1km")
        ok = img.select("QC_Day").bitwiseAnd(3).lte(1).And(raw.gt(0))
        lst = raw.updateMask(ok).multiply(0.02).subtract(273.15).rename("lst")
        return lst.copyProperties(img, ["system:time_start"])

    return ic.map(prep)


# ------------------------- year image -------------------------
def year_image(year, chirps, era5, lst, ks=ALL_K):
    out = None
    for k in ks:
        s, e = window_dates(year, k)
        c = chirps.filterDate(s, e)
        rain = c.sum().rename(f"chirps_k{k:02d}")
        dry = c.map(lambda im: im.lt(1)).sum().rename(f"dry_k{k:02d}")
        era = era5.filterDate(s, e).mean()
        era_b = era.select(["sm1", "sm2", "t2m", "vpd"]).rename(
            [f"sm1_k{k:02d}", f"sm2_k{k:02d}", f"t2m_k{k:02d}", f"vpd_k{k:02d}"]
        )
        l = lst.filterDate(s, e).mean().rename(f"lst_k{k:02d}")
        img = rain.addBands(dry).addBands(era_b).addBands(l)
        out = img if out is None else out.addBands(img)
    return out.unmask(NODATA)


def add_lonlat(f):
    xy = f.geometry().coordinates()
    return f.set({"lon": xy.get(0), "lat": xy.get(1)})


# ------------------------- outputs -------------------------
def modis_projection():
    return ee.ImageCollection("MODIS/061/MOD13Q1").first().select("NDVI").projection()


def export_drive(years, modis_proj):
    points = ee.FeatureCollection(POINTS_ASSET).map(add_lonlat)
    chirps, era5, lst = chirps_ic(), era5_ic(), lst_ic()
    for y in years:
        img = year_image(y, chirps, era5, lst)
        table = img.sampleRegions(
            collection=points, projection=modis_proj, tileScale=4, geometries=False
        ).map(lambda f, y=y: f.set("year", y))
        task = ee.batch.Export.table.toDrive(
            collection=table, description=f"climate_{y}", folder=DRIVE_FOLDER, fileFormat="CSV"
        )
        task.start()
        print(f"Started Drive export for {y} (task id: {task.id})")


def download_local(years, modis_proj, out_dir, n_chunks=N_CHUNKS):
    os.makedirs(out_dir, exist_ok=True)
    points = ee.FeatureCollection(POINTS_ASSET).randomColumn("rnd", SEED).map(add_lonlat)
    chirps, era5, lst = chirps_ic(), era5_ic(), lst_ic()
    cols = ["pid", "year", "lon", "lat"] + band_names()
    for y in years:
        final = os.path.join(out_dir, f"climate_{y}.csv")
        if os.path.exists(final):
            print(f"{y}: already done, skipping")
            continue
        img = year_image(y, chirps, era5, lst)
        parts = []
        for i in range(n_chunks):
            part = os.path.join(out_dir, f"_part_{y}_{i}.csv")
            if not os.path.exists(part):
                chunk = points.filter(
                    ee.Filter.And(ee.Filter.gte("rnd", i / n_chunks), ee.Filter.lt("rnd", (i + 1) / n_chunks))
                )
                tbl = img.sampleRegions(
                    collection=chunk, projection=modis_proj, tileScale=4, geometries=False
                ).map(lambda f, y=y: f.set("year", y))
                url = tbl.getDownloadURL(filetype="csv", selectors=cols)
                urllib.request.urlretrieve(url, part)
            parts.append(part)
            print(f"  {y}: chunk {i + 1}/{n_chunks}")
        df = pd.concat([pd.read_csv(p, dtype={"pid": str}) for p in parts], ignore_index=True)
        df = df.replace(NODATA, float("nan"))
        df.to_csv(final, index=False)
        for p in parts:
            os.remove(p)
        print(f"{y}: saved {len(df)} rows -> {final}")


# ------------------------- main -------------------------
def main(argv=None):
    ap = argparse.ArgumentParser()
    ap.add_argument("--project", required=True, help="Google Cloud project registered for Earth Engine")
    ap.add_argument("--years", nargs="*", type=int, default=DEFAULT_YEARS)
    ap.add_argument("--local-dir", default=None, help="Download CSVs here instead of Google Drive")
    ap.add_argument("--chunks", type=int, default=N_CHUNKS)
    a = ap.parse_args(argv)

    try:
        ee.Initialize(project=a.project)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=a.project)

    modis_proj = modis_projection()
    if a.local_dir:
        download_local(a.years, modis_proj, a.local_dir, a.chunks)
    else:
        export_drive(a.years, modis_proj)


if __name__ == "__main__" and "ipykernel" not in sys.modules:
    main()

In [8]:
main(["--project", "ld-slm", "--local-dir", r"crop_stress\climate"])

2001: already done, skipping
2002: already done, skipping
2003: already done, skipping
2004: already done, skipping
  2005: chunk 1/20
  2005: chunk 2/20
  2005: chunk 3/20
  2005: chunk 4/20
  2005: chunk 5/20
  2005: chunk 6/20
  2005: chunk 7/20
  2005: chunk 8/20
  2005: chunk 9/20
  2005: chunk 10/20
  2005: chunk 11/20
  2005: chunk 12/20
  2005: chunk 13/20
  2005: chunk 14/20
  2005: chunk 15/20
  2005: chunk 16/20
  2005: chunk 17/20
  2005: chunk 18/20
  2005: chunk 19/20
  2005: chunk 20/20
2005: saved 8387 rows -> crop_stress\climate\climate_2005.csv
  2006: chunk 1/20
  2006: chunk 2/20
  2006: chunk 3/20
  2006: chunk 4/20
  2006: chunk 5/20
  2006: chunk 6/20
  2006: chunk 7/20
  2006: chunk 8/20
  2006: chunk 9/20
  2006: chunk 10/20
  2006: chunk 11/20
  2006: chunk 12/20
  2006: chunk 13/20
  2006: chunk 14/20
  2006: chunk 15/20
  2006: chunk 16/20
  2006: chunk 17/20
  2006: chunk 18/20
  2006: chunk 19/20
  2006: chunk 20/20
2006: saved 8387 rows -> crop_stress\cli

C:\Users\DELL\anaconda3\envs\geospatial\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


  2024: chunk 19/20
  2024: chunk 20/20
2024: saved 8387 rows -> crop_stress\climate\climate_2024.csv


C:\Users\DELL\anaconda3\envs\geospatial\Lib\site-packages\ee\data.py:335: UserWarning: Your project has exceeded its noncommercial compute quota and is now in restricted mode. Visit https://developers.google.com/earth-engine/guides/noncommercial_tiers#restricted_mode to learn more.
  warnings.warn(


  2025: chunk 1/20
  2025: chunk 2/20
  2025: chunk 3/20
  2025: chunk 4/20
  2025: chunk 5/20
  2025: chunk 6/20
  2025: chunk 7/20
  2025: chunk 8/20
  2025: chunk 9/20
  2025: chunk 10/20
  2025: chunk 11/20
  2025: chunk 12/20
  2025: chunk 13/20
  2025: chunk 14/20
  2025: chunk 15/20
  2025: chunk 16/20
  2025: chunk 17/20
  2025: chunk 18/20
  2025: chunk 19/20
  2025: chunk 20/20
2025: saved 8387 rows -> crop_stress\climate\climate_2025.csv
